# Modeling — **IndoBERT fine-tuning** (Colab/GPU)

Driver lengkap: clone repo → fine-tune `indobert-base-p1` → tampilkan SELURUH proses
(loss/metrik per epoch, confusion matrix, contoh komentar + hasil klasifikasi).
Pilih dataset di **bagian 2** via `TAG`/`SUBSET` (default **balanced3k**).

> **Runtime → Change runtime type → GPU (T4)** sebelum mulai.

## 1. Clone repo (privat) + dependensi

In [ ]:
import os
from getpass import getpass
GH_TOKEN = os.environ.get('GH_TOKEN') or getpass('GitHub PAT (classic, scope repo): ')
![ -d jokowi_sentiment_project ] || git clone https://{GH_TOKEN}@github.com/Arachnoida/jokowi_sentiment_project.git
%cd jokowi_sentiment_project
%pip install -q 'transformers>=4.40' torch 'pymongo>=4' dnspython certifi scikit-learn matplotlib pandas numpy python-dotenv

## 2. Set MONGO_URI + jalankan fine-tuning
`processed_bert` dibaca dari Mongo Atlas (butuh IP allowlist `0.0.0.0/0`). Subset di-filter
ke 3000 baris → split kanonik 2100/600/300 (identik SVM). 4 epoch, ~4 mnt di T4.

In [ ]:
import os
from getpass import getpass
os.environ['MONGO_URI'] = os.environ.get('MONGO_URI') or getpass('MONGO_URI: ')

# === Pilih dataset ===
#   balanced 3k : TAG, SUBSET = 'balanced3k', 'outputs/labeling/balanced_3000.csv'
#   full 14k    : TAG, SUBSET = 'full14k', ''
TAG    = 'balanced3k'
SUBSET = 'outputs/labeling/balanced_3000.csv'

flags = f'--tag {TAG}' + (f' --subset {SUBSET}' if SUBSET else '')
!python -m src.modeling.train_indobert $flags

## 3. Loss & metrik per epoch
Tabel dari log Trainer (`indobert_<tag>_history.csv`): training-loss vs validation-loss,
akurasi & macro-F1 validasi tiap epoch.

In [ ]:
import pandas as pd
suffix = '' if TAG == 'full14k' else f'_{TAG}'
h = pd.read_csv(f'outputs/reports/indobert{suffix}_history.csv')
ev = h[h['eval_loss'].notna()][['epoch','eval_loss','eval_accuracy','eval_macro_f1']].round(4)
ev = ev.reset_index(drop=True)
print('Metrik validasi per epoch:')
display(ev)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,4))
if 'loss' in h.columns:
    tl = h[h['loss'].notna()]
    ax.plot(tl['epoch'], tl['loss'], marker='.', label='training loss')
ax.plot(ev['epoch'], ev['eval_loss'], marker='o', label='validation loss')
ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.set_title(f'Kurva loss — IndoBERT ({TAG})')
ax.legend(); ax.grid(alpha=.3); plt.show()

## 4. Confusion matrix + ringkasan metrik test

In [ ]:
import json
from IPython.display import Image, display
m = json.load(open(f'outputs/reports/indobert{suffix}_metrics.json'))['test']
print(f"Akurasi : {m['accuracy']}")
print(f"Macro-F1: {m['macro_f1']}")
for l in ['Negatif','Netral','Positif']:
    pc = m['per_class'][l]
    print(f"  {l:<8} P={pc['precision']:.3f} R={pc['recall']:.3f} F1={pc['f1-score']:.3f} (n={pc['support']})")
display(Image(f'outputs/reports/indobert{suffix}_test_confusion.png'))

## 5. Contoh komentar + hasil klasifikasi
Dari test set (`indobert_<tag>_predictions.csv`). ❌ = salah klasifikasi (paling penting
diperiksa — bisa jadi model salah ATAU label LLM keliru).

In [ ]:
p = pd.read_csv(f'outputs/reports/indobert{suffix}_predictions.csv')
akur = p['benar'].mean(); salah = (~p['benar']).sum()
print(f'Test: {len(p)} komentar | benar {p["benar"].sum()} | salah {salah} | akurasi {akur:.4f}')

def tampil(df, n):
    for _, r in df.head(n).iterrows():
        mark = '✅' if r['benar'] else '❌'
        t = (str(r['text'])[:110]).replace(chr(10), ' ')
        print(f"{mark} asli={r['label_asli']:<7} pred={r['prediksi']:<7} | {t}")

print(chr(10)+'— Contoh SALAH —')
tampil(p[~p['benar']], 12)
print(chr(10)+'— Contoh BENAR —')
tampil(p[p['benar']], 8)

p['teks'] = p['text'].astype(str).str.slice(0, 90)
print(chr(10)+'Tabel (20 pertama):')
display(p[['label_asli','prediksi','benar','teks']].head(20))

## 6. (Opsional) Bandingkan 3 model
Jika `svm_<tag>` & `svm_spark_<tag>` JSON sudah ada di `outputs/reports/`.

In [ ]:
need = [f'outputs/reports/svm_{TAG}_metrics.json',
        f'outputs/reports/svm_spark{suffix}_metrics.json',
        f'outputs/reports/indobert{suffix}_metrics.json']
if all(os.path.exists(p) for p in need):
    !python -m src.modeling.compare_models --tag $TAG
    display(Image(f'outputs/reports/model_comparison{"" if TAG=="full14k" else "_"+TAG}_accuracy.png'))
else:
    print('Lewati: butuh ketiga JSON. Ada:', [p for p in need if os.path.exists(p)])

In [ ]:
# Unduh artefak (untuk di-commit ke outputs/reports/ lokal)
from google.colab import files
for f in [f'indobert{suffix}_metrics.json', f'indobert{suffix}_test_confusion.png',
          f'indobert{suffix}_history.csv', f'indobert{suffix}_predictions.csv']:
    files.download(f'outputs/reports/{f}')